# 02 · Live meekijken via MQTT — De Virtuele S31-CB Klimaatsensor

In het vorige notebook zag je dat een industriële sensor (zoals de Dragino S31-CB) zijn metingen als **JSON-tekstberichten via het MQTT-protocol** naar een centrale broker kan sturen.

Omdat we geen fysieke sensoren in de klas hebben, bouwen we in dit notebook met Python een **virtuele simulatie van een stalafdeling**. Deze simulatie stuurt *exact dezelfde* JSON-berichten naar een openbare MQTT-broker op het internet. Iedereen in de klas kan zich abonneren en de gesimuleerde stal live volgen.

---

### Wikipedia-stijl diepgaande uitleg: Hoe werkt het MQTT-protocol?

**MQTT (Message Queuing Telemetry Transport)** is een lichtgewicht, netwerk-efficiënt messaging-protocol. Het is in 1999 ontwikkeld voor communicatie via satellietverbindingen met beperkte bandbreedte en hoge kosten, en is uitgegroeid tot de **wereldwijde standaard voor het Internet of Things**.

Het protocol is gebaseerd op het **Publish-Subscribe (Publiceren-Abonneren)** principe:

1. **De Broker (Centrale Server):**
   * Toestellen praten in een MQTT-netwerk **nooit rechtstreeks met elkaar**, maar communiceren altijd via een centrale server: de **broker**.
   * De broker ontvangt alle inkomende berichten, filtert ze en stuur ze door naar alle clients die daarin geïnteresseerd zijn.

2. **Topics (Onderwerpen/Adressen):**
   * Berichten worden geadresseerd aan een zogenaamd **topic**. Een topic is een tekstueel pad gescheiden door schuine strepen, bijv.: `ugent/i700301/stal/afdeling3`.
   * **Wildcards (Jokers):**
     * **`+` (Single-level wildcard):** Vervangt precies één niveau in het pad. Bijv. `ugent/i700301/stal/+` ontvangt data van `afdeling1`, `afdeling2` en `afdeling3`.
     * **`#` (Multi-level wildcard):** Vervangt alle achterliggende niveaus. Bijv. `ugent/i700301/#` ontvangt *elk* bericht dat begint met `ugent/i700301/`.

3. **Quality of Service (QoS) — Betrouwbaarheidsniveaus:**
   * **QoS 0 (At most once / Fire and forget):** Het bericht wordt maximaal één keer verstuurd zonder ontvangstbevestiging. Snelst, minste netwerkverkeer, ideaal voor continue routinemetingen.
   * **QoS 1 (At least once):** Het bericht wordt opnieuw verstuurd totdat de broker een ontvangstbevestiging geeft. Gegarandeerde aflevering, maar dubbele berichten zijn theoretisch mogelijk.
   * **QoS 2 (Exactly once):** Een vierstaps handdruk garandeert dat het bericht precies één keer wordt afgeleverd. Cruciaal voor kritieke acties (bijv. financiële transacties of kritieke alarmeringen).

4. **Retained Messages (Bewaarde Berichten):**
   * Als een zender een bericht publiceert met de vlag `retain=True`, **bewaart de broker het laatste bericht** op dat specifieke topic.
   * Wanneer een nieuwe client zich later abonneert op dat topic, ontvangt deze **onmiddellijk** dit bewaarde bericht, zonder te hoeven wachten op de volgende update.

5. **Last Will and Testament (LWT / Testamentbericht):**
   * Bij het maken van de verbinding kan een client een 'testament' opgeven bij de broker.
   * Als de verbinding onverwacht wegvalt (bijv. door een lege batterij of kabelbreuk), publiceert de broker automatisch dit testamentbericht (bijv. `status: offline`).

In [1]:
import os  # Importeer de os-module voor toegang tot omgevingsvariabelen

# Netwerk-instellingen voor de MQTT-broker
BROKER   = os.environ.get("IOT_BROKER", "broker.hivemq.com") # Adres van de centrale MQTT-server
POORT    = int(os.environ.get("IOT_POORT", "1883"))          # Standaard onversleutelde poort
KLASCODE = os.environ.get("IOT_KLASCODE", "i700301-2026-xyz") # Unieke klascode om botsingen te vermijden
GROEP    = "groep01"                                         # Unieke identificatie van jouw groep

# Stel de basis-topicstructuur samen
BASIS = f"ugent/{KLASCODE}"
print("Actieve Topicprefix voor dit notebook:", BASIS)

Actieve Topicprefix voor dit notebook: ugent/i700301-2026-xyz


## 3. Abonneren: Luister 20 seconden naar alle stalberichten

Onderstaande Python-functie maakt verbinding met de broker, abonneert zich op alle sub-topics onder `ugent/<klascode>/stal/#` en verzamelt alle live inkomende berichten gedurende een instelbaar aantal seconden.

In [2]:
import paho.mqtt.client as mqtt  # Importeer de MQTT-client module
import time                      # Voor pauzes en tijdsweergave
import json                      # Voor het verwerken van JSON-berichten

def luister(topic, seconden=20, toon=True):
    '''
    Abonneer op een specifiek MQTT `topic`, verzamel gedurende `seconden` lang 
    alle inkomende berichten en geef ze terug als een lijst van tuples: (topic, payload, retain_vlag).
    '''
    berichten = []
    
    # Callback-functie die wordt opgeroepen bij het binnenkomen van elk nieuw bericht
    def on_message(client, userdata, msg):
        payload_tekst = msg.payload.decode(errors="replace")
        berichten.append((msg.topic, payload_tekst, msg.retain))
        if toon:
            print(f"[{time.strftime('%H:%M:%S')}] {msg.topic:35s} -> {payload_tekst[:100]}")
    
    # Initialiseer de MQTT-client
    c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    c.on_message = on_message  # Koppel de callback-functie
    
    # Maak verbinding en abonneer met QoS 1
    c.connect(BROKER, POORT, keepalive=60)
    c.subscribe(topic, qos=1)
    
    # Start de netwerk-thread, wacht het opgegeven aantal seconden en koppel weer los
    c.loop_start()
    time.sleep(seconden)
    c.loop_stop()
    c.disconnect()
    
    print(f"\n{len(berichten)} bericht(en) succesvol ontvangen op topic '{topic}'.")
    return berichten

# Test de functie uit door kort te luisteren
_ = luister(f"{BASIS}/stal/#", seconden=int(os.environ.get("IOT_LUISTER", "5")))


0 bericht(en) succesvol ontvangen op topic 'ugent/i700301-2026-xyz/stal/#'.


<div style="background-color: #e8f5e9; color: #1b5e20; border-left: 5px solid #2e7d32; padding: 12px; margin: 12px 0; border-radius: 4px;">
<strong>💡 Interpretatie van de uitvoer:</strong><br>
<strong>Live MQTT Berichtstroom:</strong> Elk getoonde regel representeert een live binnenkomend MQTT-bericht.<br>• <code>[10:15:32]</code>: De exacte systeemtijd van ontvangst.<br>• <code>ugent/<klascode>/stal/afdeling1</code>: Het specifieke topic waarop het bericht is gepubliceerd.<br>• <code>{...}</code>: De nuttige inhoud (payload) in JSON-formaat, met daarin de temperatuur, luchtvochtigheid, batterijspanning en status.
</div>

## 4. Zelf een bericht publiceren

Nu gaan we zelf als 'sensor' of 'groep' een bericht sturen naar de broker. We sturen zowel een regulier bericht als een **retained statusbericht** (`status: online`).

In [3]:
# Initialiseer de MQTT-client
c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
c.connect(BROKER, POORT, keepalive=60)
c.loop_start()  # Start de netwerk-thread

# 1. Publiceer een standaard JSON-bericht op het topic van jouw groep
bericht = {"groep": GROEP, "tekst": "Hallo vanuit het notebook!", "temperatuur": 21.5}
info = c.publish(f"{BASIS}/groepen/{GROEP}", json.dumps(bericht), qos=1)
info.wait_for_publish(timeout=5)  # Wacht op de ontvangstbevestiging van de broker
print("Gepubliceerd op", f"{BASIS}/groepen/{GROEP}", "->", bericht)

# 2. Publiceer een 'retained' statusbericht op een sub-topic
# De vlag retain=True zorgt ervoor dat de broker dit bericht bewaart voor nieuwe abonnees
c.publish(f"{BASIS}/groepen/{GROEP}/status", "online", qos=1, retain=True).wait_for_publish(timeout=5)

# Verbreek de verbinding op een schone manier
c.loop_stop()
c.disconnect()

Gepubliceerd op ugent/i700301-2026-xyz/groepen/groep01 -> {'groep': 'groep01', 'tekst': 'Hallo vanuit het notebook!', 'temperatuur': 21.5}


<MQTTErrorCode.MQTT_ERR_SUCCESS: 0>

### Opdracht 4.1
Abonneer je met de functie `luister(f"{BASIS}/groepen/+/status", seconden=5)` op de statustopics van **alle groepen**. 

Wat merk je op over de berichten die je onmiddellijk binnenkrijgt, ook al publiceert er op dit exacte moment niemand actief? (Kijk naar het derde element in de uitvoer tuple: de `retain`-vlag).

In [4]:
# ==============================================================================
# OPDRACHT 4.1: RETAINED STATUSBERICHTEN ANALYSEREN
# ==============================================================================
# SCHRIJF HIER JE CODE:
# status_berichten = luister(f"{BASIS}/groepen/+/status", seconden=5)
# for topic, payload, is_retained in status_berichten:
#     print(f"Topic: {topic} | Inhoud: {payload} | Retained?: {is_retained}")


<div style="background-color: #e8f5e9; color: #1b5e20; border-left: 5px solid #2e7d32; padding: 12px; margin: 12px 0; border-radius: 4px;">
<strong>💡 Interpretatie van de uitvoer:</strong><br>
<strong>Retained Statusberichten:</strong> De output toont een Python-lijst met een tuple: <code>[('ugent/.../groepen/groep01/status', 'online', True)]</code>.<br>De waarde <code>True</code> achteraan bevestigt dat dit een **retained bericht** is. De MQTT-broker heeft dit bericht bewaard en onmiddellijk aan ons notebook geleverd zodra we ons abonneerden, zonder dat de zender op dat moment opnieuw actief hoefde te publiceren.
</div>

<details>
<summary><b>Oplossing tonen</b></summary>

De statusberichten komen **onmiddellijk** binnen, met `retain = True`: de broker bewaart per topic het laatste *retained* bericht en geeft het aan elke nieuwe abonnee. Zo weet een dashboard meteen welke sensoren "online" zijn zonder op hun volgende uplink te wachten. Een retained bericht blijft staan tot iemand een leeg retained bericht op hetzelfde topic publiceert.

</details>

## 5. De virtuele S31-CB Klimaatsensor (Simulatie)

Onderstaande code bootst **drie Dragino S31-CB sensoren** na (één voor elke stalafdeling: `afdeling1`, `afdeling2`, `afdeling3`). 

De simulatie genereert elke `INTERVAL` seconden een realistisch JSON-bericht met:
* Een licht schwankende temperatuur en luchtvochtigheid (via een *Gaussische willekeurige wandeling*).
* Een afnemende batterijspanning.
* Een **alarmvlag** (`"interrupt": 1`) zodra de temperatuur $\ge 30 \ ^\circ	ext{C}$ of de luchtvochtigheid $\ge 90\%$ wordt.

In [6]:
import random             # Voor het genereren van willekeurige temperatuurschommelingen
import datetime as dt     # Voor het aanmaken van UTC-tijdstempels

INTERVAL = float(os.environ.get("IOT_INTERVAL", "2"))   # Seconden tussen twee opeenvolgende uplinks
DUUR     = float(os.environ.get("IOT_DUUR", "10"))      # Totale duur dat de simulatie draait (s)

# Starttoestand per stalafdeling: temperatuur (°C), vochtigheid (% RV), batterij (V) en IMEI-nummer
afdelingen = {
    "afdeling1": {"T": 22.0, "RV": 60.0, "bat": 3.52, "imei": "868508065628110"},
    "afdeling2": {"T": 23.5, "RV": 65.0, "bat": 3.41, "imei": "868508065628111"},
    "afdeling3": {"T": 27.8, "RV": 72.0, "bat": 3.29, "imei": "868508065628112"},
}

def s31cb_json(naam, s, historiek):
    '''Genereer een JSON-structuur conform de Dragino S31-CB specificatie.'''
    nu = dt.datetime.now(dt.timezone.utc)
    bericht = {
        "IMEI": s["imei"], "IMSI": "206010000000000", "Model": "S31x-CB (Simulatie)",
        "temperature": round(s["T"], 1), "humidity": round(s["RV"], 1),
        "interrupt": 0, "interrupt_level": 0, "count": 0,
        "battery": round(s["bat"], 2), "signal": random.randint(14, 26),
        "time": nu.strftime("%Y/%m/%d %H:%M:%S"),
        "latitude": 0.0, "longitude": 0.0, "gps_time": "1970-01-01T00:00:00Z",
    }
    # Voeg de laatste 8 metingen als historiek toe aan de JSON-payload
    for i, (T, RV, t) in enumerate(historiek[-8:][::-1], start=1):
        bericht[str(i)] = [T, RV, t]
    return bericht

# Maak een simulatie MQTT-client aan
sensor_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=f"s31cb-sim-{random.randint(0,9999)}")
sensor_client.connect(BROKER, POORT, keepalive=60)
sensor_client.loop_start()

hist = {n: [] for n in afdelingen}
start_tijd = time.time()

print(f"Virtuele sensoren starten publicatie gedurende {DUUR:.0f}s op {BASIS}/stal/<afdeling> ...")

while time.time() - start_tijd < DUUR:
    for naam, s in afdelingen.items():
        # Genereer kleine willekeurige temperatuur- en vochtigheidsvariaties (Gaussische verdeling)
        s["T"]   = round(min(max(s["T"]  + random.gauss(0, 0.3), 15), 35), 1)
        s["RV"]  = round(min(max(s["RV"] + random.gauss(0, 1.0), 30), 99), 1)
        s["bat"] = max(s["bat"] - 0.0005, 2.6)  # Simuleer een hele lichte batterijontlading
        
        hist[naam].append([s["T"], s["RV"], dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")])
        bericht = s31cb_json(naam, s, hist[naam])
        
        # Schakel het alarm in als de drempelwaarden overschreden worden
        if s["T"] >= 30 or s["RV"] >= 90:
            bericht["interrupt"] = 1
            
        # Publiceer het JSON-bericht op het topic van de specifieke afdeling
        sensor_client.publish(f"{BASIS}/stal/{naam}", json.dumps(bericht), qos=1)
        
    print(f"[{time.strftime('%H:%M:%S')}] " + "  ".join(f"{n}: {s['T']} °C / {s['RV']} %" for n, s in afdelingen.items()))
    time.sleep(INTERVAL)

sensor_client.loop_stop()
sensor_client.disconnect()
print("Simulatie gestopt.")

Virtuele sensoren starten publicatie gedurende 10s op ugent/i700301-2026-xyz/stal/<afdeling> ...
[09:00:06] afdeling1: 22.4 °C / 61.2 %  afdeling2: 23.1 °C / 65.4 %  afdeling3: 28.0 °C / 70.8 %


[09:00:08] afdeling1: 22.2 °C / 60.2 %  afdeling2: 23.2 °C / 64.5 %  afdeling3: 28.0 °C / 70.6 %


[09:00:10] afdeling1: 22.2 °C / 59.1 %  afdeling2: 23.3 °C / 66.1 %  afdeling3: 28.2 °C / 70.7 %


[09:00:12] afdeling1: 21.9 °C / 57.4 %  afdeling2: 23.5 °C / 66.1 %  afdeling3: 27.7 °C / 70.5 %


[09:00:14] afdeling1: 21.8 °C / 55.3 %  afdeling2: 23.7 °C / 65.4 %  afdeling3: 27.9 °C / 69.8 %


Simulatie gestopt.


## 6. Live stalgegevens opvangen, verwerken en visualiseren

In deze stap vangen we de live verzonden gegevens op, verwerken we de JSON-berichten naar een overzichtelijke **Pandas DataFrame** (tabel) en tekenen we de verloopgrafiek per stalafdeling.

In [7]:
import pandas as pd            # Voor tabelverwerking
import matplotlib.pyplot as plt  # Voor het maken van grafieken

# Luister live naar alle afdelingen onder ugent/<klascode>/stal/+
ruwe_berichten = luister(f"{BASIS}/stal/+", seconden=int(os.environ.get("IOT_LUISTER2", "5")), toon=False)

rijen = []
for topic, payload_tekst, retain in ruwe_berichten:
    try:
        b = json.loads(payload_tekst)  # Zet JSON-tekst om naar Python dictionary
    except json.JSONDecodeError:
        continue                       # Sla berichten over die geen geldige JSON zijn
        
    rijen.append({
        "afdeling":      topic.split("/")[-1],
        "tijd":          pd.to_datetime(b["time"], format="%Y/%m/%d %H:%M:%S", utc=True),
        "temperatuur_C": b["temperature"],
        "RV_pct":        b["humidity"],
        "batterij_V":    b["battery"],
        "alarm":         bool(b.get("interrupt", 0))
    })

# Maak een Pandas DataFrame aan
df = pd.DataFrame(rijen)

if len(df):
    print("Samenvatting per stalafdeling (gemiddelde, min, max):")
    print(df.groupby("afdeling")[["temperatuur_C", "RV_pct"]].agg(["mean", "min", "max"]).round(1))
    print("\nLaatste 5 ontvangen metingen:")
    print(df.tail())
else:
    print("Geen berichten ontvangen — controleer of de simulatie draait.")


0 bericht(en) succesvol ontvangen op topic 'ugent/i700301-2026-xyz/stal/+'.
Geen berichten ontvangen — controleer of de simulatie draait.


<div style="background-color: #e8f5e9; color: #1b5e20; border-left: 5px solid #2e7d32; padding: 12px; margin: 12px 0; border-radius: 4px;">
<strong>💡 Interpretatie van de uitvoer:</strong><br>
<strong>Pandas Aggregatie-tabel (DataFrame):</strong> De opgevangen JSON-berichten zijn omgezet naar een overzichtelijke tabel. Met behulp van <code>groupby('afdeling').agg(['mean', 'min', 'max'])</code> berekent Pandas automatisch de gemiddelde, minimale en maximale temperatuur en vochtigheid per stalafdeling over de gehele luisterperiode.
</div>

In [8]:
if len(df):
    fig, ax = plt.subplots(figsize=(9, 4))
    
    # Teken een temperatuurlijn voor elke afzonderlijke afdeling
    for naam, groep in df.groupby("afdeling"):
        ax.plot(groep["tijd"], groep["temperatuur_C"], marker="o", label=naam)
        
    # Teken een rode stippellijn voor de alarmdrempel op 30 °C
    ax.axhline(30, color="tab:red", linestyle="--", linewidth=1.5, label="Alarmdrempel (30 °C)")
    
    ax.set_ylabel("Temperatuur (°C)")
    ax.set_title("Live Stal-temperatuur via MQTT")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

## 7. Zonder Python: Live meekijken via de Browser

Je kunt hetzelfde MQTT-verkeer ook live volgen via een webpagina in de browser met behulp van de **HiveMQ WebSocket Client**:
<https://www.hivemq.com/demos/websocket-client/>

1. Vul in: Host `broker.hivemq.com`, Port `8884`, vink **SSL** aan en klik op **Connect**.
2. Klik bij *Subscriptions* op **Add New Topic Subscription** en voer in: `ugent/<jouw-klascode>/stal/#`.
3. Klik op **Subscribe**. Je ziet nu alle live berichten binnenstromen!

---

### Opdrachten

1. **Alarm-berichten filteren:** Schrijf een Python-expressie om enkel de berichten uit de opgevangen lijst te filteren waar `"interrupt": 1` staat.
2. **Last Will & Testament (LWT):** Zoek in de paho-mqtt documentatie op hoe je met `client.will_set()` een afscheidsbericht instelt. Waarom is dit voor een stalsensor van levensbelang?
3. **Klimaatcomputer-sturing & QoS:** Welke QoS-klasse (0, 1 of 2) kies je voor (a) routinematige temperatuurmetingen, en (b) een kritiek temperatuuralarm? Motiveer je keuze.
4. **Topic-architectuur ontwerpen:** Ontwerp een overzichtelijke topic-structuur voor een landbouwbedrijf met 2 stallen, elk 4 afdelingen, met sensoren voor temperatuur, RV, CO₂ en NH₃. Hoe abonneer je je in één keer op alle NH₃-sensoren van het hele bedrijf?

In [9]:
# ==============================================================================
# OPDRACHT SECTIE 7: ALARMBERICHTEN FILTEREN & LAST WILL (LWT)
# ==============================================================================
# SCHRIJF HIER JE CODE VOOR OPDRACHT 1:
# alarm_berichten = [
#     (topic.split("/")[-1], json.loads(payload)["temperature"], json.loads(payload)["humidity"])
#     for topic, payload, retain in ruwe_berichten
#     if json.loads(payload).get("interrupt") == 1
# ]

# SCHRIJF HIER JE ANTWOORD VOOR OPDRACHT 2:
# lwt_uitleg = ""


<details>
<summary><b>Oplossing tonen</b></summary>

1. Meestal **afdeling3** (start op 27,8 °C, drempel 30 °C); de alarmvlag `interrupt` wordt 1 zodra T ≥ 30 °C of RV ≥ 90 %.
2. `client.will_set(topic, payload="offline", qos=1, retain=True)` vóór `connect()`. De broker publiceert dit zelf als de client wegvalt zonder nette `disconnect()`. Voor een stalsensor is dat cruciaal: een uitgevallen sensor (lege batterij, kabelbreuk, stroomuitval) wordt zo *actief* gemeld; bij een webwinkel is een verbroken verbinding van één gebruiker niet relevant.
3. Meting: QoS 0 of 1 (een gemiste meting is niet erg, over 10 min komt de volgende; QoS 0 bespaart batterij). Alarm: QoS 1 of 2 (mag nooit verloren gaan; QoS 2 vermijdt dubbele alarmen, maar kost twee extra berichten).
4. Bijvoorbeeld `bedrijf/stal<1-2>/afdeling<1-4>/<grootheid>` met grootheid `temperatuur`, `rv`, `co2`, `nh3`. Alle NH₃ in één keer: `bedrijf/+/+/nh3`. Alles van stal 2: `bedrijf/stal2/#`.

</details>